In [ ]:
import os
import csv
import time
import datetime as dt
import praw
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import networkx as nx
from networkx.algorithms import community, cuts

# Set Reddit API credentials (replace with your own)
os.environ["REDDIT_CLIENT_ID"] = "INSERT_CLIENT_ID"
os.environ["REDDIT_CLIENT_SECRET"] = "INSERT_CLIENT_SECRET"
os.environ["REDDIT_USER_AGENT"] = "sna-project"

# Define crawling time window
AFTER  = "2025-09-27"
BEFORE = "2025-10-28"

# Output file
OUTCSV = "seed_homes_dynamic.csv"

# Crawling parameters
SLEEP = 0.4
PROGRESS_EVERY = 2000
MAX_ROWS = None  # set a limit (e.g., 300_000) if needed

# Political subreddits (home communities)
HOMES = [
    "Socialism","LateStageCapitalism","Anarchism","SocialDemocracy","progressive","AskALiberal",
    "Conservative","Libertarian","AskConservatives","Capitalism","ConservativeMemes"
]

# Convert date string to Unix timestamp
def to_epoch(s):
    return int(
        dt.datetime.strptime(s, "%Y-%m-%d")
        .replace(tzinfo=dt.timezone.utc)
        .timestamp()
    )

a, b = to_epoch(AFTER), to_epoch(BEFORE)

# Initialize Reddit client
reddit = praw.Reddit(
    client_id=os.environ["REDDIT_CLIENT_ID"],
    client_secret=os.environ["REDDIT_CLIENT_SECRET"],
    user_agent=os.environ.get("REDDIT_USER_AGENT", "sna-project homes dynamic"),
    check_for_async=False,
)

# Define output schema
fields = [
    "id","author","subreddit","link_id","parent_id","body",
    "created_utc","day","score","permalink","source"
]

# Create CSV file with header
with open(OUTCSV, "w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=fields).writeheader()

total = 0
t0 = time.time()

# Crawl each subreddit
for sub_name in HOMES:

    print(f"\n=== r/{sub_name} [home] ===")

    posts = 0
    sr = reddit.subreddit(sub_name)

    for submission in sr.new(limit=None):

        ts = int(getattr(submission, "created_utc", 0) or 0)

        if ts >= b:
            continue

        if ts < a:
            break

        posts += 1

        # Load all comments in the thread
        submission.comments.replace_more(limit=0)

        for comment in submission.comments.list():

            body = (getattr(comment, "body", "") or "").strip()

            # Skip deleted/empty comments
            if body in ("[deleted]", "[removed]", ""):
                continue

            author = str(getattr(comment, "author", "") or "")

            # Skip invalid authors
            if not author or author.lower() == "automoderator":
                continue

            created_utc = int(getattr(comment, "created_utc", 0) or 0)

            # Keep only comments within the time window
            if not (a <= created_utc < b):
                continue

            row = {
                "id": comment.id,
                "author": author,
                "subreddit": sub_name,
                "link_id": f"t3_{submission.id}",
                "parent_id": getattr(comment, "parent_id", ""),
                "body": body,
                "created_utc": created_utc,
                "day": dt.datetime.utcfromtimestamp(created_utc).strftime("%Y-%m-%d"),
                "score": getattr(comment, "score", 0),
                "permalink": f"https://www.reddit.com{getattr(comment, 'permalink', '')}",
                "source": "home",
            }

            # Append row to CSV
            with open(OUTCSV, "a", newline="", encoding="utf-8") as f:
                csv.DictWriter(f, fieldnames=fields).writerow(row)

            total += 1

            # Optional early stop
            if MAX_ROWS and total >= MAX_ROWS:
                print(f"🛑 MAX_ROWS={MAX_ROWS:,} reached.")
                break

            # Progress logging
            if total % PROGRESS_EVERY == 0:
                elapsed = time.time() - t0
                print(f"{total:,} rows | r/{sub_name} | posts={posts:,} | {elapsed/60:.1f} min")

        time.sleep(SLEEP)

    if MAX_ROWS and total >= MAX_ROWS:
        break

print(f"\n✅ Crawling completed: {total:,} rows → {OUTCSV}")

In [ ]:

SEED_CSV = "seed_homes_dynamic.csv"
LABELS_CSV = "labels_homes.csv"
SEED_LABELED_OUT = "seed_homes_labeled.csv"

# load
seed = pd.read_csv(SEED_CSV)
seed["author"] = seed["author"].astype(str).str.strip()

# counter per (author, subreddit)
cnt = (seed.groupby(["author","subreddit"])
             .agg(n_comments=("id","count"),
                  sum_score=("score","sum"),
                  last_ts=("created_utc","max"))
             .reset_index())

# top subreddit for authot con tie-break
cnt = cnt.sort_values(
    by=["author","n_comments","sum_score","last_ts","subreddit"],
    ascending=[True, False, False, False, True]  
top = cnt.drop_duplicates("author", keep="first").rename(columns={"subreddit":"home_label"})

# save mapping autore→home_label
top[["author","home_label","n_comments","sum_score","last_ts"]].to_csv(LABELS_CSV, index=False)

# applica al seed per comodo (seed etichettato)
seed_labeled = seed.merge(top[["author","home_label"]], on="author", how="left")
seed_labeled.to_csv(SEED_LABELED_OUT, index=False)

print(f"✔️ Salvato mapping → {LABELS_CSV} (utenti: {top.shape[0]:,})")
print(f"✔️ Seed etichettato → {SEED_LABELED_OUT} (righe: {seed_labeled.shape[0]:,})")

# mini report
rep = (top["home_label"].value_counts().rename_axis("home_label").reset_index(name="n_users"))
print(rep.head(10).to_string(index=False))

In [ ]:
AFTER  = "2025-09-27"
BEFORE = "2025-10-28"
LABELS_CSV = "labels_homes.csv"
OUTCSV   = "arena_activity_dynamic.csv"

SLEEP = 0.4
PROGRESS_EVERY = 2000
MAX_ROWS = None  # es. 300_000

ARENAS = [
    "worldnews", "news", "geopolitics",
    "NeutralPolitics", "PoliticalDiscussion",
    "IsraelPalestine", "MiddleEast", "politics"
]
# <<< fine >>>

def to_epoch(s): 
    return int(dt.datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=dt.timezone.utc).timestamp())
a,b = to_epoch(AFTER), to_epoch(BEFORE)

# mapping author→home_label
lab = pd.read_csv(LABELS_CSV)
lab["author"] = lab["author"].astype(str).str.strip()
author2label = dict(zip(lab["author"], lab["home_label"]))

reddit = praw.Reddit(
    client_id=os.environ["REDDIT_CLIENT_ID"],
    client_secret=os.environ["REDDIT_CLIENT_SECRET"],
    user_agent=os.environ.get("REDDIT_USER_AGENT","sna-project arena dynamic"),
    check_for_async=False,
)

fields = ["id","author","subreddit","link_id","parent_id","body","created_utc","day","score","permalink","home_label","source"]
with open(OUTCSV, "w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=fields).writeheader()

total = 0
t0 = time.time()
for sub_name in ARENAS:
    print(f"\n=== r/{sub_name} [arena] ===")
    posts = 0
    sr = reddit.subreddit(sub_name)
    for s in sr.new(limit=None):
        ts = int(getattr(s,"created_utc",0) or 0)
        if ts >= b: 
            continue
        if ts < a: 
            break
        posts += 1
        s.comments.replace_more(limit=0)
        for c in s.comments.list():
            au = str(getattr(c,"author","") or "")
            if not au or au.lower()=="automoderator": 
                continue
            if au not in author2label:         # SOLO utenti seed etichettati
                continue
            cu = int(getattr(c,"created_utc",0) or 0)
            if not (a <= cu < b): 
                continue
            body = (getattr(c,"body","") or "").strip()
            if body in ("[deleted]","[removed]",""):
                continue

            row = {
                "id": c.id,
                "author": au,
                "subreddit": sub_name,
                "link_id": f"t3_{s.id}",
                "parent_id": getattr(c,"parent_id",""),
                "body": body,
                "created_utc": cu,
                "day": dt.datetime.utcfromtimestamp(cu).strftime("%Y-%m-%d"),
                "score": getattr(c,"score",0),
                "permalink": f"https://www.reddit.com{getattr(c,'permalink','')}",
                "home_label": author2label[au],
                "source": "arena",
            }
            with open(OUTCSV, "a", newline="", encoding="utf-8") as f:
                csv.DictWriter(f, fieldnames=fields).writerow(row)
            total += 1

            if MAX_ROWS and total >= MAX_ROWS:
                print(f"🛑 MAX_ROWS={MAX_ROWS:,} raggiunto.")
                break

            if total % PROGRESS_EVERY == 0:
                el = time.time()-t0
                print(f"{total:,} rows | r/{sub_name} | posts={posts:,} | {el/60:.1f}m")
        time.sleep(SLEEP)
    if MAX_ROWS and total >= MAX_ROWS:
        break

print(f"\n✅ Arena completata: {total:,} righe → {OUTCSV}")

In [ ]:
import os
import pandas as pd

# Input file paths (update if needed)
SEED_CSV   = "seed_homes_dynamic.csv"      # output from home crawling
LABELS_CSV = "labels_homes.csv"            # author -> home_label (majority)
ARENA_CSV  = "arena_activity_dynamic.csv"  # output from arena crawling
OUT_CSV    = "combined_dynamic.csv"

# Target schema (minimum useful union of columns)
TARGET_COLS = [
    "id","author","subreddit","link_id","parent_id",
    "body","created_utc","day","score","permalink",
    "home_label","source"
]


def load_if_exists(path):
    """
    Load CSV if it exists and apply basic normalization.
    Returns None if file is missing.
    """
    if not os.path.exists(path):
        print(f"⚠️  File not found: {path}")
        return None

    df = pd.read_csv(path)

    # Normalize author field
    if "author" in df.columns:
        df["author"] = df["author"].astype(str).str.strip()

    # Ensure timestamp is numeric
    if "created_utc" in df.columns:
        df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")

    # Create 'day' column if missing
    if "day" not in df.columns and "created_utc" in df.columns:
        df["day"] = pd.to_datetime(
            df["created_utc"], unit="s", errors="coerce"
        ).dt.strftime("%Y-%m-%d")

    # Cast key columns to string
    for col in ("link_id","parent_id","subreddit","source","home_label"):
        if col in df.columns:
            df[col] = df[col].astype(str)

    return df


# Load datasets
seed   = load_if_exists(SEED_CSV)
arena  = load_if_exists(ARENA_CSV)
labels = load_if_exists(LABELS_CSV)

if seed is None and arena is None:
    raise SystemExit("❌ No datasets available to merge (seed and arena missing).")


# Build author -> label mapping (ground truth)
author2label = {}

if labels is not None and {"author","home_label"}.issubset(labels.columns):

    author2label = dict(
        labels.dropna(subset=["author","home_label"])
              .drop_duplicates("author")[["author","home_label"]]
              .values
    )

else:
    print("ℹ️  LABELS_CSV missing or invalid. home_label may remain empty.")


def align_and_label(df, fallback_source=None):
    """
    Align dataset to target schema and assign labels.
    """
    if df is None:
        return None

    # Assign source if missing
    if "source" not in df.columns and fallback_source is not None:
        df["source"] = fallback_source

    # Ensure home_label column exists
    if "home_label" not in df.columns:
        df["home_label"] = pd.NA

    # Fill missing labels using mapping
    if author2label:
        mask_empty = (
            df["home_label"].isna() |
            df["home_label"].astype(str).str.lower().isin(["", "nan"])
        )
        df.loc[mask_empty, "home_label"] = df.loc[mask_empty, "author"].map(author2label)

    # Ensure all target columns exist
    for col in TARGET_COLS:
        if col not in df.columns:
            df[col] = pd.NA

    return df[TARGET_COLS].copy()


# Align datasets
seed_aligned  = align_and_label(seed,  fallback_source="home")
arena_aligned = align_and_label(arena, fallback_source="arena")


# Merge datasets
dfs = [d for d in [seed_aligned, arena_aligned] if d is not None]
combined = pd.concat(dfs, ignore_index=True)


# Data cleaning

# Remove rows with missing key fields
combined = combined.dropna(subset=["author","link_id","created_utc"]).copy()

# Remove duplicate comments
combined = combined.drop_duplicates(subset=["id"]).copy()


# Save final dataset
combined.to_csv(OUT_CSV, index=False)


# Quick summary statistics
def nunique_safe(df, col):
    return df[col].nunique(dropna=True) if col in df.columns else 0


print(f"✅ Saved dataset: {OUT_CSV}")
print(f"Total rows: {len(combined):,}")
print(f"Unique users: {nunique_safe(combined,'author'):,}")
print(f"Unique threads: {nunique_safe(combined,'link_id'):,}")

print("\nSource distribution:")
print(combined["source"].value_counts(dropna=False).to_string())

if "home_label" in combined.columns:
    print("\nUsers with assigned home_label:",
          combined.loc[combined["home_label"].notna(),"author"].nunique())
    print("Rows without home_label:", combined["home_label"].isna().sum())

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import community, cuts

GRAPH_DIR = "daily_graphs"
OUT_CSV = "home_label_metrics_daily.csv"

rows = []

# Iterate over daily graph files (.gexf)
for fname in sorted(os.listdir(GRAPH_DIR)):

    if not fname.endswith(".gexf"):
        continue

    # Extract day from filename (expected format: G_YYYY-MM-DD.gexf)
    day = fname.replace("G_", "").replace(".gexf", "")
    path = os.path.join(GRAPH_DIR, fname)

    G = nx.read_gexf(path)

    # Skip empty graphs
    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        print(f"{day}: empty graph, skipping.")
        continue

    # Extract node attribute: home_label
    labels = nx.get_node_attributes(G, "home_label")

    # Normalize labels (handle missing / invalid values)
    norm_labels = {}
    for node, lbl in labels.items():
        if lbl is None:
            norm_labels[node] = None
        else:
            s = str(lbl)
            norm_labels[node] = None if s.lower() in {"nan", "none", ""} else s

    # Build groups of nodes by home_label
    groups = {}
    for node, lbl in norm_labels.items():
        if lbl is None:
            continue
        groups.setdefault(lbl, []).append(node)

    # Compute global modularity across label-based partitions
    partitions = [set(nodes) for nodes in groups.values() if len(nodes) >= 2]

    if partitions and G.number_of_edges() > 0:
        try:
            Q = community.quality.modularity(G, partitions)
        except Exception:
            Q = np.nan
    else:
        Q = np.nan

    # Compute conductance for each group
    for lbl, nodes in groups.items():

        if len(nodes) < 2:
            continue

        try:
            phi = cuts.conductance(G, set(nodes))
        except Exception:
            phi = np.nan

        rows.append({
            "day": day,
            "home_label": lbl,
            "size": len(nodes),
            "conductance": phi,
            "modularity_global": Q
        })

    print(f"{day} → modularity={Q:.3f} | groups analyzed={len(groups)}")


# Convert to DataFrame and save
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f"\n✅ Saved: {OUT_CSV}")
print(df.head())

In [ ]:


df = pd.read_csv("home_label_metrics_daily.csv")
df = df.sort_values("day")

# formattiamo le date in oggetti datetime
df["day"] = pd.to_datetime(df["day"])
labels = sorted(df["home_label"].unique())

# divisione in due gruppi (come prima)
half = len(labels)//2
groups = [labels[:half], labels[half:]]

for i, subset in enumerate(groups, 1):
    fig, ax = plt.subplots(figsize=(10,5))
    for lbl, g in df[df["home_label"].isin(subset)].groupby("home_label"):
        ax.plot(g["day"], g["conductance"], marker='o', label=lbl)
    
    # personalizza asse X: giorno + mese
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))  # es. 02-Oct
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))    # tick ogni 5 giorni
    
    plt.title(f"Conduttanza giornaliera (gruppo {i})")
    plt.xlabel("Data")
    plt.ylabel("Conductance (φ)")
    plt.grid(alpha=0.3)
    plt.legend(title="Home label", bbox_to_anchor=(1.05,1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
rep = pd.read_csv("daily_graphs_report.csv").sort_values("day")
rep["day"] = pd.to_datetime(rep["day"])

plt.figure(figsize=(9,4))
plt.plot(rep["day"], rep["assortativity_home"], color='firebrick', marker='o')
plt.title("Assortatività per home_label nel tempo")
plt.xlabel("Data")
plt.ylabel("Assortatività (r)")
plt.grid(alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
plt.tight_layout()
plt.show()